In [1]:
from czi_to_zarr import czi_to_omezarr_folder, single_nd2_to_zarr, organize_czi_multipos_folder

In [3]:
path = r"P:\Kai\2021-09-23_KSB014_bead_DC_tests\2021-09-23_KSB014_5FOVs_565_100pct_647_0.15umZ.nd2"
out_path = r"P:\Kai\2021-09-23_KSB014_bead_DC_tests\2021-09-23_KSB014_5FOVs_565_100pct_647_0.15umZ_images"
single_nd2_to_zarr(path, out_path)

  0%|          | 0/5 [00:00<?, ?it/s]

Found files of shape: (5, 5, 2, 68, 2304, 2304)


100%|██████████| 5/5 [21:11<00:00, 254.21s/it]


In [8]:
organize_czi_multipos_folder(r'P:\Kai\2021-10-07_KSB015_1D_HKRad21AID_4x10pos_aux_elyra\images')

In [2]:
input_folders = [r'P:\Kai\2021-10-07_KSB015_1D_HKRad21AID_4x10pos_aux_elyra\images']
out_path = r'P:\Kai\2021-10-07_KSB015_1D_HKRad21AID_4x10pos_aux_elyra\zarr_images'

czi_to_omezarr_folder(input_folders, out_path, continue_from='P0028')

  0%|          | 0/45 [00:00<?, ?it/s]

Found files of shape: (20, 2, 65, 1280, 1280)
27 0
27 1
27 2
27 3
27 4
27 5
27 6
27 7
27 8
27 9
27 10
27 11
27 12
27 13
27 14
27 15
27 16
27 17
27 18
27 19
27 20
27 21
27 22
27 23
27 24
27 25
27 26


[Parallel(n_jobs=-1)]: Using backend ThreadingBackend with 8 concurrent workers.
[Parallel(n_jobs=-1)]: Done   2 tasks      | elapsed:  1.4min
[Parallel(n_jobs=-1)]: Done   8 out of  20 | elapsed:  1.7min remaining:  2.6min
[Parallel(n_jobs=-1)]: Done  11 out of  20 | elapsed:  2.7min remaining:  2.2min
[Parallel(n_jobs=-1)]: Done  14 out of  20 | elapsed:  2.8min remaining:  1.2min
[Parallel(n_jobs=-1)]: Done  17 out of  20 | elapsed:  3.5min remaining:   36.5s
[Parallel(n_jobs=-1)]: Done  20 out of  20 | elapsed:  3.5min remaining:    0.0s
[Parallel(n_jobs=-1)]: Done  20 out of  20 | elapsed:  3.5min finished
 62%|██████▏   | 28/45 [03:38<02:12,  7.81s/it][Parallel(n_jobs=-1)]: Using backend ThreadingBackend with 8 concurrent workers.
[Parallel(n_jobs=-1)]: Done   2 tasks      | elapsed:  1.3min
[Parallel(n_jobs=-1)]: Done   8 out of  20 | elapsed:  1.6min remaining:  2.4min
[Parallel(n_jobs=-1)]: Done  11 out of  20 | elapsed:  2.6min remaining:  2.2min
[Parallel(n_jobs=-1)]: Done  

In [2]:
import czifile
img = czifile.imread(r"P:\Kai\2021-11-05_KSB020_HKWT_red_beads_dc_test\2021-11-05_KSB020_red_bead_dc_test_5pos_5loops.czi")

img.shape

(1, 1, 5, 2, 5, 51, 1280, 1280, 1)

In [3]:
img = img[0,0,:,:,:,:,:,:,0]
img.shape

(5, 2, 5, 51, 1280, 1280)

In [4]:
import numpy as np
img = np.moveaxis(img, 2, 1)
img.shape

(5, 5, 2, 51, 1280, 1280)

In [5]:
import napari
napari.view_image(img)

Viewer(axes=Axes(visible=False, labels=True, colored=True, dashed=False, arrows=True), camera=Camera(center=(0.0, 639.5, 639.5), zoom=0.8623534010946051, angles=(0.0, 0.0, 90.0), perspective=0.0, interactive=True), cursor=Cursor(position=(1.0, 1.0, 0.0, 0.0, 0.0, 0.0), scaled=True, size=1, style=<CursorStyle.STANDARD: 'standard'>), dims=Dims(ndim=6, ndisplay=2, last_used=5, range=((0.0, 4.0, 1.0), (0.0, 4.0, 1.0), (0.0, 1.0, 1.0), (0.0, 50.0, 1.0), (0.0, 1279.0, 1.0), (0.0, 1279.0, 1.0)), current_step=(0, 0, 0, 0, 0, 0), order=(0, 1, 2, 3, 4, 5), axis_labels=('0', '1', '2', '3', '4', '5')), grid=GridCanvas(stride=1, shape=(-1, -1), enabled=False), layers=[<Image layer 'img' at 0x1517e3fed00>], scale_bar=ScaleBar(visible=False, colored=False, ticks=True, position=<Position.BOTTOM_RIGHT: 'bottom_right'>, font_size=10.0, unit=None), text_overlay=TextOverlay(visible=False, color=array([0.5, 0.5, 0.5, 1. ]), font_size=10.0, position=<TextOverlayPosition.TOP_LEFT: 'top_left'>, text=''), help

In [7]:
import zarr
from numcodecs import Blosc

P, T, C, Z, Y, X = img.shape
compressor = Blosc(cname='zstd', clevel=5, shuffle=Blosc.BITSHUFFLE)
chunks = (1,1,1,Y,X)

for i, pos in enumerate(['P'+str(i).zfill(4) for i in range(P)]):
    store = zarr.DirectoryStore(r'P:\Kai\2021-11-05_KSB020_HKWT_red_beads_dc_test\zarr_images\\'+pos)
    root = zarr.group(store=store, overwrite=True)

    root.attrs['multiscale'] = {'multiscales': [{'version': '0.2', 'name': 'dataset', 'datasets': [{'path': '0'}]}]}

    multiscale_level = root.create_dataset(name = str(0), compressor=compressor, shape=(T, C, Z, Y, X), chunks=chunks)
    multiscale_level[:] = img[i]

In [1]:
from czi_to_zarr import single_nd2_to_zarr
input_path = r"P:\Kai\2021-11-05_KSB020_multichannel_bead_dc_test\2021-11-05_KSB020_1D-E_8pos_5loops.nd2"
out_path = r'P:\Kai\2021-11-05_KSB020_multichannel_bead_dc_test\zarr_images_test'
single_nd2_to_zarr(input_path, out_path, continue_from = None)

  0%|          | 0/8 [00:00<?, ?it/s]

Found files of shape: (8, 5, 2, 54, 2304, 2304)


100%|██████████| 8/8 [30:26<00:00, 228.29s/it]


In [8]:
from nd2reader import ND2Reader
import json
import datetime
from dask import delayed

with ND2Reader(r"P:\Kai\2021-08-19_MT001_NikonTiE2_HKWT_2E_MitoTrace\21-08-19_MT001_Ibidi1E_stack005.nd2") as images:
    images.bundle_axes = 'czyx'
    images.iter_axes = 'vt'
    sample_shape = images.sizes
    X = sample_shape['x']
    Y = sample_shape['y']
    Z = sample_shape['z']
    C = sample_shape['c']
    T = sample_shape['t']
    P = sample_shape['v']
    s = (P, T, C, Z, Y, X)
    print(len(images), images.frame_shape)
        


336 (2, 64, 2304, 2304)


In [9]:
12*28

336

In [16]:
-1 in (-1,)

True